In [1]:
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
openai_client = OpenAI()

In [4]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful humorous assistant"
    },
    {
        "role": "user",
        "content": "Say Hello World!"
    }
]

In [5]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.7
)

In [6]:
print(response.choices[0].message.content)

Hello, World! 🌍🎉 It's great to see you here! What’s on your mind today?


# Small mini project to talk with AI assistant using OpenAI client

In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [8]:
load_dotenv()

True

In [9]:
def main():
    if not os.environ.get("OPENAI_API_KEY"):
        print("OPENAI key missing!")
        return
    client = OpenAI()
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful, encouraging, and highly intelligent Study Buddy. "
                "Your goal is to help the user understand complex topics using the Socratic method. "
                "Instead of giving answers directly, explain concepts simply and ask guiding questions "
                "to test their understanding. Keep responses under 4 sentences.")
        }
    ]
    while True:
        user_input = input("You: ").lower().strip()
        if user_input in ["bye", "quit", "exit"]:
            print("Goodbye!")
            break
        messages.append({"role": "user", "content": user_input})
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.7)
        ai_response = response.choices[0].message.content
        print(f"AI: {ai_response}\n")
        messages.append({"role": "system", "content": ai_response})

In [10]:
main()

You:  hi


AI: Hello! How can I assist you with your studies today? What topic are you curious about?



You:  Retrieval Augmented Generation


AI: Great topic! Retrieval Augmented Generation (RAG) combines information retrieval with generative models. Can you think of how using external information might enhance the generation of responses?



You:  bye


Goodbye!


# mini project #2: Website summariser!

In [11]:
import os
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

In [12]:
def extract_text_from_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        # Raise an exception for bad status codes (404, 500, etc.)
        response.raise_for_status() 
        
        # Parse HTML using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Remove non-content elements like scripts, styles, navbars, and footers
        for element in soup(["script", "style", "nav", "footer", "header", "aside"]):
            element.decompose()
            
        # Extract text and clean up whitespace
        text = soup.get_text(separator=' ')
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        clean_text = ' '.join(chunk for chunk in chunks if chunk)
        return clean_text

    except Exception as e:
        print(f"Error fetching or parsing the URL: {e}")
        return None

In [13]:
def summarize_content(web_text):
    client = OpenAI()
    
    # Truncate text to save context window tokens
    max_characters = 40000 # (gpt-4o-mini supports up to 128k, but keeping it short helps in speed/cost)
    truncated_text = web_text[:max_characters]
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert research assistant. Analyze the provided website text "
                "and generate a clear, concise summary. Highlight the primary purpose of the site, "
                "key takeaways, and main bullet points."
            )
        },
        {
            "role": "user",
            "content": f"Please summarize the following website content:\n\n{truncated_text}"
        }
    ]
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error during OpenAI API call: {e}"


In [14]:
def main(url):
    try:
        extracted_text = extract_text_from_url(url)
        summary = summarize_content(extracted_text)
        return summary
        
    except error as e:
        return f"Error during Summarising {url}: {e}"

In [15]:
print(main("https://en.wikipedia.org/wiki/BMW_X7"))

**Summary of BMW X7 - Wikipedia**

**Primary Purpose of the Site:**
The Wikipedia page provides comprehensive information about the BMW X7, a full-size luxury crossover SUV, including its specifications, design, production history, and various models.

**Key Takeaways:**
- The BMW X7 is BMW's largest and second most expensive SUV, introduced as a flagship model.
- It features a spacious interior with standard third-row seating and advanced technology.
- The X7 is produced in multiple countries, including the United States, Thailand, and India.
- A facelift was introduced in 2022, featuring design updates and new technology.

**Main Bullet Points:**
- **Overview:**
  - Manufacturer: BMW
  - Model Code: G07
  - Production: December 2018 – present
  - Class: Full-size luxury crossover SUV
  - Layout: Front-engine, all-wheel drive (xDrive)

- **Design and Features:**
  - Designer: Julien Sarreméjean
  - Platform: Cluster Architecture (CLAR)
  - Dimensions: Length ranges from 5,151 to 5,181